# ⚡ Módulo 10 — Orchestration: Concurrent

> **Objetivo:** Ejecutar varios agentes en paralelo sobre la misma tarea y agregar resultados.

📚 **Doc oficial:** [Concurrent Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/concurrent?pivots=programming-language-python)

## ¿Qué es Concurrent Orchestration?

Todos los agentes procesan el **mismo input al mismo tiempo** y un agregador combina sus salidas.

```
                ┌─→ [Researcher] ─┐
Usuario  ───────┼─→ [Marketer]   ─┼──→  Aggregator  ──→  AgentResponse final
                └─→ [Legal]      ─┘
```

Ideal para:
- Brainstorming desde múltiples perspectivas
- Ensemble reasoning / voting
- Análisis multi-disciplinario (legal + técnico + marketing)

## API clave

```python
from agent_framework.orchestrations import ConcurrentBuilder

workflow = ConcurrentBuilder(participants=[a, b, c]).build()
events = await workflow.run("...")
outputs = events.get_outputs()     # [AgentResponse]  — un solo objeto con N mensajes
```

> El agregador por defecto produce **un `AgentResponse`** con un mensaje por participante.
> Puedes pasar un agregador custom con `.with_aggregator(callback)`.


## ⚙️ Setup

In [7]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


✅ Helpers cargados.


## 1️⃣ Fan-out básico con 3 expertos

Tres agentes con expertise distinta analizan el mismo prompt **en paralelo**.
El framework hace fan-out, espera a todos y agrega.


In [ ]:
from agent_framework import AgentResponse
from agent_framework.orchestrations import ConcurrentBuilder

client = create_chat_client()

researcher = Agent(
    client=client,
    name="researcher",
    instructions=(
        "Eres un investigador experto de mercado y productos. Dado un prompt, proporciona información concisa, "
        "información factual, oportunidades y riesgos. Responde en español."
    ),
)

marketer = Agent(
    client=client,
    name="marketer",
    instructions=(
        "Eres un estratega de marketing creativo. Elabora propuestas de valor convincentes y "
        "mensajes dirigidos alineados al prompt. Responde en español."
    ),
)

legal = Agent(
    client=client,
    name="legal",
    instructions=(
        "Eres un revisor cauteloso de aspectos legales y cumplimiento. Destaca restricciones, descargos de responsabilidad, "
        "y preocupaciones de políticas basadas en el prompt. Responde en español."
    ),
)

workflow = ConcurrentBuilder(participants=[researcher, marketer, legal]).build()

events = await workflow.run(
    "Estamos lanzando una nueva bicicleta eléctrica económica para viajeros urbanos."
)

# get_outputs() retorna una lista; outputs[0] es UN solo AgentResponse
# cuyo .messages contiene un mensaje por participante (con .author_name).
outputs = events.get_outputs()

if outputs:
    print("===== Final Aggregated Results =====")
    final: AgentResponse = outputs[0]
    for msg in final.messages:
        name = msg.author_name or "assistant"
        print(f"{'-' * 60}\n\n[{name}]:\n{msg.text}\n")

===== Final Aggregated Results =====

------------------------------------------------------------
[assistant]:
**Información Concisa:**  
El producto es una bicicleta eléctrica económica dirigida al mercado de viajeros urbanos, un segmento en crecimiento debido a la demanda de alternativas de transporte sostenibles, económicas y eficientes para distancias cortas y medianas en entornos urbanos.

**Información Factual:**  
1. **Demanda creciente:** El mercado global de bicicletas eléctricas se valoró en aproximadamente 48,000 millones de dólares en 2022 y se proyecta un crecimiento anual de alrededor del 12% hasta 2030.  
2. **Segmento urbano:** Los viajeros urbanos buscan vehículos compactos, de bajo costo operativo y que permitan esquivar el tráfico. Las bicicletas eléctricas son populares especialmente en ciudades con problemas de congestión vial, tarifas de estacionamiento altas y sistemas de transporte insuficientes.  
3. **Competencia:** Marcas como Xiaomi, Rad Power Bikes y Lectr

## 2️⃣ Agregador custom — síntesis con LLM

Por defecto recibes los outputs concatenados. Si quieres que **otro LLM resuma** o consolide
todos los outputs en un solo texto, pasa un callback async a `.with_aggregator(...)`.


In [ ]:
from agent_framework import AgentExecutorResponse

summarizer_agent = Agent(
    client=client,
    name="summarizer",
    instructions=(
        "Consolidas los resultados de múltiples expertos de dominio en un resumen cohesivo y conciso "
        "con conclusiones claras. Mantenlo en menos de 200 palabras. Responde en español."
    ),
)


async def summarize_results(results: list[AgentExecutorResponse]) -> str:
    """Toma los outputs de todos los expertos y los pasa al summarizer."""
    sections: list[str] = []
    for r in results:
        messages = getattr(r.agent_response, "messages", [])
        final_text = messages[-1].text if messages and hasattr(messages[-1], "text") else "(no content)"
        sections.append(f"{r.executor_id}:\n{final_text}")

    prompt = "\n\n".join(sections)
    response = await summarizer_agent.run(prompt)
    return response.messages[-1].text if response.messages else ""


workflow = (
    ConcurrentBuilder(participants=[researcher, marketer, legal])
    .with_aggregator(summarize_results)
    .build()
)

output_text = None
async for event in workflow.run(
    "Estrategia de lanzamiento para una nueva bicicleta eléctrica",
    stream=True,
):
    if event.type == "output":
        output_text = event.data

print("===== Final Consolidated Output =====\n")
print(output_text)


## 3️⃣ Streaming + intermediate outputs

Por defecto sólo el agregador emite `"output"`. Si quieres ver el output de cada participante
**a medida que llega**, usa `intermediate_output_from=[...]` y observa los eventos `"intermediate"`.


In [ ]:
from agent_framework import AgentResponseUpdate

workflow = ConcurrentBuilder(
    participants=[researcher, marketer, legal],
    intermediate_output_from="all_other",
).build()

last_author = None
async for event in workflow.run(
    "Analiza nuestra nueva estrategia de lanzamiento de producto.",
    stream=True,
):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = getattr(update, "author_name", None) or event.type
        if author != last_author:
            if last_author is not None:
                print()
            print(f"[{author}]: {update.text}", end="", flush=True)
            last_author = author
        else:
            print(update.text, end="", flush=True)
print()


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Fan-out paralelo | `ConcurrentBuilder(participants=[a, b, c]).build()` |
| Agregador custom | `.with_aggregator(async_callback)` |
| Outputs intermedios | `intermediate_output_from=[...]` |
| Streaming | `async for event in workflow.run(prompt, stream=True):` |

**Siguiente módulo →** [Módulo 11 — Orchestration Handoff](./11_orchestration_handoff.ipynb)
